# 01. データ収集 — 人間プレイの記録

Super Mario Bros (`SuperMarioBros-1-1-v0`) を実際にプレイして、フレーム画像とアクションを記録します。

## キー操作

| キー | 動作 |
|---|---|
| `d` | 右 |
| `a` | 左 |
| `s` | しゃがむ (下) |
| `w` | 上 |
| `o` | ジャンプ (A) |
| `p` | ダッシュ / ファイア (B) |
| Enter | スタート |
| Space | セレクト |

セルを上から順に実行してください。最後のセルを実行するとゲーム画面のウィンドウが開き、プレイできます。

- ミスして死んだりゴールしたりすると、`play_human` の仕様により自動的に次のエピソードが始まります。
- **Escキーでプレイを終了**すると、その時点までのデータも保存されます。

In [ ]:
import gym_super_mario_bros
import nes_py.play as nes_play
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT
from nes_py._image_viewer import ImageViewer
from nes_py.wrappers import JoypadSpace

ENV_ID = "SuperMarioBros-1-1-v0"

# play_human が開くプレイ画面のウィンドウは、デフォルトではNES本来の解像度 (256x240) で
# 小さく表示される。RENDER_SCALE 倍のウィンドウで開くよう、内部のビューワー生成処理を差し替える。
RENDER_SCALE = 2


def _make_keyboard_viewer_scaled(env, keys_to_action):
    relevant_keys = {key for combo in keys_to_action.keys() for key in combo}
    height, width = env.observation_space.shape[0], env.observation_space.shape[1]
    caption = env.spec.id if env.spec is not None else env.__class__.__name__
    return ImageViewer(
        caption,
        height * RENDER_SCALE,
        width * RENDER_SCALE,
        monitor_keyboard=True,
        relevant_keys=relevant_keys,
    )


nes_play._make_keyboard_viewer = _make_keyboard_viewer_scaled


def action_name(action_idx: int) -> str:
    """SIMPLE_MOVEMENT のインデックスをファイル名に使える文字列に変換する"""
    buttons = SIMPLE_MOVEMENT[action_idx]
    return "_".join(buttons) if buttons else "NOOP"


env = JoypadSpace(gym_super_mario_bros.make(ENV_ID, render_mode="human"), SIMPLE_MOVEMENT)

print("SIMPLE_MOVEMENT:", SIMPLE_MOVEMENT)
print("reset() ->", env.reset())

In [ ]:
# 記録用ヘルパー: プレイ中はメモリ上にバッファし、エピソード終了時にまとめて画像として書き出す
import csv
import time
from pathlib import Path

from PIL import Image

DATA_DIR = Path("../data")
EPISODES_DIR = DATA_DIR / "episodes"
MANIFEST_PATH = DATA_DIR / "manifest.csv"
EPISODES_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_FIELDS = [
    "episode_id", "num_frames", "score", "x_pos_max", "coins",
    "flag_get", "time_remaining", "truncated", "timestamp",
]

episode_idx = 0
frames = []
actions = []


def _append_manifest_row(row: dict) -> None:
    file_exists = MANIFEST_PATH.exists()
    with open(MANIFEST_PATH, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=MANIFEST_FIELDS)
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


def _flush_episode(truncated: bool) -> None:
    """バッファ内のフレームを画像として保存し、manifest.csv に1行追記する"""
    global episode_idx
    if not frames:
        return

    ep_dir = EPISODES_DIR / f"ep_{episode_idx:04d}"
    ep_dir.mkdir(parents=True, exist_ok=True)
    for i, (frame, action) in enumerate(zip(frames, actions)):
        filename = f"{i:06d}_a{action}_{action_name(action)}.png"
        Image.fromarray(frame).save(ep_dir / filename)

    row = {
        "episode_id": ep_dir.name,
        "num_frames": len(frames),
        "score": env.unwrapped._score,
        "x_pos_max": env.unwrapped._x_position_max,
        "coins": env.unwrapped._coins,
        "flag_get": env.unwrapped._flag_get,
        "time_remaining": env.unwrapped._time,
        "truncated": truncated,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    _append_manifest_row(row)
    print(
        f"[{ep_dir.name}] frames={row['num_frames']} score={row['score']} "
        f"x_pos_max={row['x_pos_max']} flag_get={row['flag_get']} truncated={truncated}"
    )

    episode_idx += 1
    frames.clear()
    actions.clear()


def recording_callback(state, action, reward, done, next_state):
    frames.append(state)
    actions.append(action)
    if done:
        _flush_episode(truncated=False)

In [ ]:
from nes_py.play import play_human

try:
    play_human(env, callback=recording_callback)
finally:
    # Escキー等で途中終了した場合、残っているデータを truncated=True として保存する
    _flush_episode(truncated=True)